<table>
<tr><td><img style="height: 150px;" src="images/geo_hydro1.jpg"></td>
<td bgcolor="#FFFFFF">
    <p style="font-size: xx-large; font-weight: 900; line-height: 100%">pyKOLK</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);"><b style=color:red;>Solution</b> pockets</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);">Georg Kaufmann</p>
    </td>
<td><img style="height: 150px;" src="images/pyKOLK.png"></td>
</tr>
</table>

----
# `pyKOLK`

pyKOLK, a program package prediction the shape of solution pockets in caves from pre-defined climate conditions.

In [1]:
import numpy as np
import scipy.interpolate
import matplotlib.pyplot as plt
import sys
import libCHEM,libKOLK

----
# Solution pocket -flow model-
In this notebook, we create all function to model the shape of a solution pocket, driven by pre-defined climate variables.

----
## Growth -flow model-

### Flow rate
The second defining property is the **volumetic flow rate** of the drops, given as:
$$
Q = \frac{V}{\tau}
$$
with $V$ [m$^3$] the volume of the drop ($V \sim 0.1$ cm$^3$),
$\tau$ [s] the drop interval.


### Growth along solution pocket
To predict the new retreat layer removed from the cave surface, a **retreat rate** is
defined as:
$$
R_i = \frac{m_{CaCO_3}}{\rho_{CaCO_3}} F_i
$$
with 
$m_{CaCO_3}$ [kg/mol] the molar mass of calcite,
$\rho_{CaCO_3}$ [kg/m$^3$] the density of calcite.
The flux rates along the profile need to be calculated now from the flow model ...

### Flow model
We calculate **film thickness** $\delta_i$ [m],
**flow velocity** $v_i$ [m/s]
**travel time** $t_i$ [s],
**calcium concentration** $c_i$ [mol/m$^3$], and
**flux rate** $F_i$ [mol/m$^2$/s] from:
$$\begin{array}{rcl}
\delta_i &=& \left( \frac{3\eta}{\rho g} \frac{Q}{w_i \sin\gamma_i}  \right)^\frac{1}{3} \\
v_i      &=& \left( \frac{\rho g}{3\eta} \frac{Q^2 \sin\gamma_i}{w_i^2} \right)^\frac{1}{3} \\
t_i      &=& t_{i-1} + \Delta t_i = t_{i-1} + \frac{l_i}{v_i} \\
c_i      &=& c_{i-1} - \frac{\alpha \Delta t_i \left( c_{in}-c_{eq} \right)}{\delta_i} \\
F_i      &=& F_{i-1} \left( 1 - \frac{l_i w_i \alpha}{Q} \right) = F_{i-1} \left( 1 - \frac{\alpha \Delta t_i}{\delta_i} \right)
\end{array}$$

Here, $l_i$ [m] is the length segment along the surface,
$l_i=\sqrt{(x_{i}-x_{i-1})^2 + (y_{i}-y_{i-1})^2}$,
and the angle $\gamma_i$ is given as $\gamma_i=\sin^{-1}\left( \frac{y_{i-1}-y_{i}}{l_i} \right)$

Then the new retreated surface is calculated as
$$\begin{array}{rcl}
x_i &=& x_i + R_i \sin\gamma_i \\
y_i &=& y_i + R_i \cos\gamma_i \\
\end{array}$$

### Functions
The following `python` functions will be used to calculate the solution pocket:

In [ ]:
def runKolk_flow(infile1='KOLK_parameter.in',infile2='KOLK_timeline.in',path='work/',mix=1.0,climate='simple',plot=False):
    """
    pyKOLK
    function creates solution pocket morphologies in 2D,
    using the flow version for shape
    and time as master variable (simple as argument)
    or temperature as master variable (temp as argument)
    adopted from fortran90 code ...
    """
    # fixed parameter values
    dropvolume  = 1e-7    # drop volume [m^3]
    Vdrop       = 0.1e-6  # drop volume [m3]
    rhoWater    = 1000.   # density water [kg/m3]
    g           = 9.81    # grav acceleration [m/s2]
    etaWater    = 1.14e-3 # viscosity water [Pas]
    mCaCO3      = 0.1001  # molar mass calcite [kg/mol]
    rhoCaCO3    = 2700.   # density calcite [kg/m3]
    year2sec    = 365.25*24*60*60
    # read input data
    sidex,nx,init_height,timeStep,timeWrite,TSoilmin,TSoilmax,PSoilmin,PSoilmax, \
                PAtmmin,PAtmmax,TCavemin,TCavemax,PCavemin,PCavemax,dropCavemin,dropCavemax = \
                readParameter2D(infile=infile1,path=path,control=True)
    timeStart,timeEnd,rawTimeline = readTimeline2D(infile=infile2,path=path,control=True)
    # create grid
    x,y,dx = createGrid2D(sidex,nx,init_height,plot=True)
    # define stack for geometry, fill with initial shape
    nstack          = 50
    kolkGeom        = np.zeros(nx*2*nstack).reshape(nx,2,nstack)
    kolkGeom[:,0,0] = x
    kolkGeom[:,1,0] = y
    kolkSave        = np.zeros(nstack)
    kolkSave[0]     = timeStart
    # initialize fields along surface
    growth = np.zeros(len(x))
    angle  = np.zeros(len(x))
    width  = np.zeros(len(x))
    flux   = np.zeros(len(x))
    film   = np.zeros(len(x))
    vel    = np.zeros(len(x))
    c      = np.zeros(len(x))
    t      = np.zeros(len(x))
    #---------------------
    # time loop
    #---------------------
    iSaved   = 0
    itime    = 0
    time     = timeStart + timeStep
    timeSave = timeStart + timeWrite
    print('Start time loop ...')
    while (time <= timeEnd):
        # interpolate climate parameter
        if (climate == 'simple'):
            Tsoil = np.interp(time,[timeStart,timeEnd],[TSoilmin,TSoilmax])
            Tcave = np.interp(time,[timeStart,timeEnd],[TCavemin,TCavemax])
            Psoil = np.interp(time,[timeStart,timeEnd],[PSoilmin,PSoilmax])
            Pcave = np.interp(time,[timeStart,timeEnd],[PCavemin,PCavemax])
            Dcave = np.interp(time,[timeStart,timeEnd],[dropCavemin,dropCavemax])
        elif (climate == 'seasonT'):
            Tsoil = TSoilmin + (TSoilmax-TSoilmin)*0.5 * (1. - np.cos(2*np.pi*time))
            Tcave = np.interp(time,[timeStart,timeEnd],[TCavemin,TCavemax])
            Psoil = np.interp(time,[timeStart,timeEnd],[PSoilmin,PSoilmax])
            Pcave = np.interp(time,[timeStart,timeEnd],[PCavemin,PCavemax])
            Dcave = np.interp(time,[timeStart,timeEnd],[dropCavemin,dropCavemax])
        elif (climate == 'seasonP'):
            Tsoil = np.interp(time,[timeStart,timeEnd],[TSoilmin,TSoilmax])
            Tcave = np.interp(time,[timeStart,timeEnd],[TCavemin,TCavemax])
            Psoil = PSoilmin + (PSoilmax-PSoilmin)*0.5 * (1. - np.cos(2*np.pi*time))
            Pcave = np.interp(time,[timeStart,timeEnd],[PCavemin,PCavemax])
            Dcave = np.interp(time,[timeStart,timeEnd],[dropCavemin,dropCavemax])
        elif (climate == 'paleo'):
            Tsoil = np.interp(time,rawTimeline[:,0],rawTimeline[:,1])
            Tcave = TCavemin+ (TCavemax-TCavemin) * (TSoil-TSoilmin)/(TSoilmax-TSoilmin)
            Psoil = PSoilmin+ (PSoilmax-PSoilmin) * (TSoil-TSoilmin)/(TSoilmax-TSoilmin)
            Pcave = PCavemin+ (PCavemax-PCavemin) * (TSoil-TSoilmin)/(TSoilmax-TSoilmin)
            Dcave = DCavemin+ (DCavemax-DCavemin) * (TSoil-TSoilmin)/(TSoilmax-TSoilmin)
        Q = Vdrop / Dcave
        # drop chemistry
        CEQopen   = libCHEM.CEQ_limestone_open(Tsoil,Psoil/1e6)
        CEQclosed = libCHEM.CEQ_limestone_closed(Tsoil,Psoil/1e6)
        Cin       = mix*CEQopen + (1-mix)*CEQclosed
        CEQcave   = libCHEM.CEQ_limestone_open(Tcave,Pcave/1e6)
        # calcium concentration, flux rate, and growth rate
        c[0]      = Cin
        flux[0]   = libCHEM.FCaCO3(Cin,CEQcave)
        if (flux[0]<0): 
            flux[0]=0.
        growth[0] = mCaCO3 / rhoCaCO3 * flux[0] * timeStep * year2sec
        # refine grid    
        if (x[1] > 1.5*dx):
            x,y = refineGrid2D(x,y)
        # loop along surface
        angle[0]  = 0.
        for i in range(1,len(x)):
            dl       = np.sqrt( (x[i-1]-x[i])**2 + (y[i-1]-y[i])**2)
            width[i] = 2.*np.pi*x[i]
            angle[i] = np.arcsin((y[i-1]-y[i])/dl)
            if (angle[i] > 0.):
                film[i]  = np.cbrt(3*etaWater/(rhoWater*g) * Q / (width[i]*np.sin(angle[i])))
                vel[i]   = np.cbrt(rhoWater*g/(3*etaWater) * Q**2*np.sin(angle[i])/(width[i]**2))
                t[i]     = t[i-1] + dl/vel[i]
                c[i]     = c[i-1] + flux[i-1]/vel[i]/film[i]*dl
            else:
                t[i]     = t[i-1]
                c[i]     = c[i-1]
            flux[i]   = libCHEM.FCaCO3(c[i],CEQcave)
            if (flux[i]<0): 
                flux[i]=0.
            growth[i] = mCaCO3 / rhoCaCO3 * flux[i] * timeStep * year2sec
        # update shape
        x[0]      = x[0]
        y[0]      = y[0] + growth[0]
        for i in range(1,len(x)):
            x[i] = x[i] + growth[i]*np.sin(angle[i])
            y[i] = y[i] + growth[i]*np.cos(angle[i])
        vel[0]  = vel[1]
        film[0] = film[1]
        # save stalagmite shape for defined time step
        if (time >= timeSave):
            iSaved = iSaved + 1
            if (iSaved > kolkGeom.shape[2]-1):
                print('iSaved too large')
                sys.exit()
            print('%30s %10i %10.2f %10.2f' % ('Time step, time [a], TSoil [C]:',iSaved,time,Tsoil))
            kolkGeom[:,0,iSaved] = x
            kolkGeom[:,1,iSaved] = y
            kolkSave[iSaved]     = time
            timeSave            += timeWrite
        # update time
        time += timeStep
        itime += 1
    print('End time loop ...')
    # plot solution pocket
    if (plot):
        plotKolk(kolkGeom,kolkSave,sidex,iSaved)
    return kolkGeom,kolkSave,iSaved,sidex

----